In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q polars numpy scipy cupy-cuda12x tqdm

In [ ]:
#========================================
# $1 Configuration (CẬP NHẬT)
#========================================
import os
import time
import logging
import numpy as np
import polars as pl
import scipy.sparse as sp
from dataclasses import dataclass
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.INFO, force=True, format='%(asctime)s - %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class UserProfileConfig:
    """Hyperparameters configuration for User Profile Generation."""
    version: str           = 'v1'
    dim: int               = 384
    chunk_size: int        = 1_000_000
    use_gpu: bool          = True

    # --- CẤU HÌNH TÊN CỘT DỮ LIỆU ---
    col_user: str          = "user_id"
    col_item: str          = "book_id"   # <-- Chắc chắn là book_id

    base_dir: str          = '/content/drive/My Drive/Thesis/book_recsys'

    path_interactions: str = ""
    path_item_mapping: str = ""
    path_item_vectors: str = ""
    path_out_user_mapping: str = ""
    path_out_user_vectors: str = ""

    def __post_init__(self):
        data_dir = f"{self.base_dir}/data"
        self.path_interactions = f"{data_dir}/processed/eval/interactions_train.csv"
        self.path_item_mapping = f"{data_dir}/processed/test/cb_sbert_{self.version}_book_index.csv"
        self.path_item_vectors = f"{data_dir}/processed/test/cb_sbert_{self.version}_matrix.npy"

        out_dir = f"{data_dir}/processed/test"
        os.makedirs(out_dir, exist_ok=True)
        self.path_out_user_mapping = f"{out_dir}/cb_sbert_{self.version}_user_index.csv"
        self.path_out_user_vectors = f"{out_dir}/cb_sbert_{self.version}_user_profiles_mean.npy"

        # [MỚI] Đường dẫn lưu file History Master
        self.path_out_user_history = f"{out_dir}/cb_sbert_{self.version}_user_histories.parquet"

In [ ]:
#=================================
# $2 Core Architecture Layer
#=================================

class ItemProfile:
    """Single Source of Truth cho dữ liệu Sách (Items)."""
    def __init__(self, mapping_path: str, vectors_path: str, vector_dim: int = 384):
        self.mapping_path = mapping_path
        self.vectors_path = vectors_path
        self.vector_dim = vector_dim

        self._item_id_to_idx = {}
        self._item_vectors = None
        self.total_items = 0

    def load(self) -> None:
        if not os.path.exists(self.mapping_path):
            raise FileNotFoundError(f"Không tìm thấy Mapping: {self.mapping_path}")

        df = pl.read_csv(self.mapping_path)
        col = next((c for c in ["item_id", "book_id", "id"] if c in df.columns), df.columns[0])

        item_ids = df[col].cast(pl.Utf8).to_list()
        self._item_id_to_idx = {i: idx for idx, i in enumerate(item_ids)}
        self.total_items = len(item_ids)

        if not os.path.exists(self.vectors_path):
            raise FileNotFoundError(f"Không tìm thấy Vectors: {self.vectors_path}")

        self._item_vectors = np.load(self.vectors_path, mmap_mode='r')
        logger.info(f"[ItemProfile] Đã nạp thành công {self.total_items:,} items.")

    def get_all_vectors(self) -> np.ndarray:
        return self._item_vectors[:]

class MeanProfileManager:
    """Máy làm toán cho thuật toán Mean Profile."""
    def __init__(self, use_gpu: bool = False, vector_dim: int = 384):
        self.vector_dim = vector_dim
        self.use_gpu = use_gpu

        if use_gpu:
            try:
                import cupy as cp
                import cupyx.scipy.sparse as cp_sparse
                self.xp = cp
                self.xps = cp_sparse
                logger.info("Chế độ tính toán GPU (CuPy) Đã Kích Hoạt.")
            except ImportError:
                logger.warning("Không tìm thấy CuPy. Chuyển về CPU.")
                self.use_gpu = False

        if not self.use_gpu:
            self.xp = np
            self.xps = sp
            logger.info("Chế độ tính toán CPU (SciPy) Đã Kích Hoạt.")

In [ ]:
#=================================
# $3 Data Pipeline Layer
#=================================
import gc

def prepare_user_mapping(config: UserProfileConfig) -> dict:
    """Quét dữ liệu để lập chỉ mục User ID -> Index."""
    logger.info("Đang tạo User Index Mapping từ Interactions...")
    df_users = (
        pl.scan_csv(config.path_interactions)
        .select(config.col_user)
        .unique()
        .collect()
    )

    user_ids = df_users[config.col_user].cast(pl.Utf8).to_list()
    user_mapping = {u_id: idx for idx, u_id in enumerate(user_ids)}

    # Lưu Mapping để dùng cho bước Đánh giá/Production
    pl.DataFrame({
        "row_idx": np.arange(len(user_ids)),
        "user_id": user_ids
    }).write_csv(config.path_out_user_mapping)

    logger.info(f"Phát hiện tổng cộng {len(user_ids):,} Users. Đã lưu Mapping.")
    return user_mapping

def build_profiles_streaming(
    config: UserProfileConfig,
    manager: MeanProfileManager,
    item_profile: ItemProfile,
    user_mapping: dict
) -> tuple:
    """
    Toán học Ma trận Streaming (MapReduce):
    Vừa tính toán Mean Vector, vừa đóng gói Lịch sử tương tác.
    """
    num_users = len(user_mapping)
    num_items = item_profile.total_items

    # 1. Chuẩn bị nguyên liệu trên RAM/VRAM
    engine_item_vectors = manager.xp.asarray(item_profile.get_all_vectors())

    # Kho chứa tổng vector và tổng số lượt tương tác
    total_sum_profiles = manager.xp.zeros((num_users, config.dim), dtype=manager.xp.float32)
    total_interaction_counts = manager.xp.zeros(num_users, dtype=manager.xp.float32)

    # Kho chứa lịch sử (History Accumulator): {user_id: [item_id1, item_id2, ...]}
    user_history_accumulator = {u_id: [] for u_id in user_mapping.keys()}

    # 2. Thiết lập Lazy Reader chỉ lấy các cột cần thiết
    lazy_interactions = (
        pl.scan_csv(config.path_interactions)
        .select([config.col_user, config.col_item])
    )

    total_rows = lazy_interactions.select(pl.len()).collect().item()
    total_chunks = int(np.ceil(total_rows / config.chunk_size))

    logger.info(f"Bắt đầu xử lý {total_rows:,} interactions ({total_chunks} chunks)...")
    item_mapping_fast = item_profile._item_id_to_idx

    # 3. Vòng lặp Chunking
    chunk_iter = tqdm(range(0, total_rows, config.chunk_size), desc="Streaming Matrix & History")

    for offset in chunk_iter:
        # Đọc một đoạn dữ liệu
        chunk_df = lazy_interactions.slice(offset, config.chunk_size).collect()

        valid_u_idx = []
        valid_i_idx = []

        # Duyệt qua từng dòng trong chunk
        for u_id, i_id in zip(chunk_df[config.col_user], chunk_df[config.col_item]):
            u_id_str = str(u_id)
            i_id_str = str(i_id)

            # Chỉ xử lý nếu User và Book đều tồn tại trong Catalog
            if u_id_str in user_mapping and i_id_str in item_mapping_fast:
                u_idx = user_mapping[u_id_str]
                i_idx = item_mapping_fast[i_id_str]

                valid_u_idx.append(u_idx)
                valid_i_idx.append(i_idx)

                # Gom nhóm lịch sử trực tiếp
                user_history_accumulator[u_id_str].append(i_id_str)

        if not valid_u_idx:
            continue

        # Chuyển chỉ số lên thiết bị (CPU/GPU)
        device_rows = manager.xp.asarray(valid_u_idx, dtype=manager.xp.int32)
        device_cols = manager.xp.asarray(valid_i_idx, dtype=manager.xp.int32)
        device_vals = manager.xp.ones(len(valid_u_idx), dtype=manager.xp.float32)

        # Tạo ma trận thưa CSR cho Chunk này
        R_chunk = manager.xps.csr_matrix(
            (device_vals, (device_rows, device_cols)),
            shape=(num_users, num_items)
        )

        # NHÂN MA TRẬN & CỘNG DỒN
        # (Users x Items) @ (Items x Dim) -> (Users x Dim)
        total_sum_profiles += R_chunk.dot(engine_item_vectors)
        total_interaction_counts += manager.xp.array(R_chunk.sum(axis=1)).flatten()

        # Giải phóng bộ nhớ tạm
        del chunk_df, valid_u_idx, valid_i_idx, device_rows, device_cols, device_vals, R_chunk
        if manager.use_gpu:
            manager.xp.get_default_memory_pool().free_all_blocks()
        gc.collect()

    # 4. CHIA TRUNG BÌNH ĐỂ RA MEAN PROFILE
    # Tránh chia cho 0 cho những user không có tương tác hợp lệ
    safe_counts = manager.xp.maximum(total_interaction_counts, 1.0)
    final_profiles = total_sum_profiles / safe_counts.reshape(-1, 1)

    if manager.use_gpu:
        final_profiles = manager.xp.asnumpy(final_profiles)

    # Trả về bộ đôi: Ma trận vector và Từ điển lịch sử
    return final_profiles, user_history_accumulator

In [ ]:
#=================================
# $4 Orchestrator Main Loop Pipeline
#=================================

def main_build_pipeline(config: UserProfileConfig):
    start_time = time.time()
    logger.info("=========== USER PROFILE PIPELINE INITIALIZED ===========")

    # 1. Nạp Nguồn chân lý duy nhất (Items/Books)
    item_profile = ItemProfile(
        mapping_path=config.path_item_mapping,
        vectors_path=config.path_item_vectors,
        vector_dim=config.dim
    )
    item_profile.load()

    # 2. Khởi tạo Động cơ toán học (Hỗ trợ GPU/CPU)
    manager = MeanProfileManager(use_gpu=config.use_gpu, vector_dim=config.dim)

    # 3. Quét dữ liệu để tạo Mapping User ID
    user_mapping = prepare_user_mapping(config)

    # 4. Thực hiện Streaming để tính Vector và gom nhóm Lịch sử
    # Hàm này trả về: (Matrix, History Dictionary)
    user_profiles_matrix, history_dict = build_profiles_streaming(
        config=config,
        manager=manager,
        item_profile=item_profile,
        user_mapping=user_mapping
    )

    # 5. Lưu trữ Artifacts (Export)

    # A. Lưu Ma trận Vector (Dùng cho FAISS tìm kiếm)
    logger.info(f"Đang ghi File Numpy Matrix: {user_profiles_matrix.shape}...")
    np.save(config.path_out_user_vectors, user_profiles_matrix)

    # B. Lưu Lịch sử Master (Dùng cho Masking/Lọc sách đã đọc)
    logger.info("Đang đóng gói Lịch sử vào file Parquet...")

    # Chuyển Dictionary {user_id: [item_ids]} sang Polars DataFrame
    # Chúng ta lọc set() ở đây để đảm bảo mỗi cuốn sách chỉ xuất hiện 1 lần trong history của user
    history_df = pl.DataFrame({
        "user_id": list(history_dict.keys()),
        "history_item_ids": [list(set(items)) for items in history_dict.values()]
    })

    # Ghi file Parquet với nén Snappy để tối ưu tốc độ đọc/ghi
    history_df.write_parquet(config.path_out_user_history, compression="snappy")

    # Giải phóng bộ nhớ
    del user_profiles_matrix, history_dict, history_df
    gc.collect()

    elapsed = time.time() - start_time
    logger.info(f"====== PIPELINE HOÀN TẤT - TỔNG THỜI GIAN: {elapsed:.2f}s ======")
    logger.info(f"File Vectors: {config.path_out_user_vectors}")
    logger.info(f"File History: {config.path_out_user_history}")

# Kích hoạt thực thi
if __name__ == "__main__":
    app_config = UserProfileConfig()
    main_build_pipeline(app_config)

In [ ]:
#=================================
# $5 Acceptance Framework
#=================================

def acceptance_test(config: UserProfileConfig):
    logger.info("--- Initiating Acceptance Test Routine (Integrity Assurance) ---")

    if not os.path.exists(config.path_out_user_vectors) or not os.path.exists(config.path_out_user_mapping):
        logger.error("[FAIL] Không tìm thấy các file Artifact đầu ra.")
        return

    # Kiểm tra Mapping
    df_users = pl.read_csv(config.path_out_user_mapping)
    total_users_csv = df_users.height
    logger.info(f"[PASS] User Mapping ghi nhận {total_users_csv:,} Users.")

    # Kiểm tra Ma trận Vectors
    matrix = np.load(config.path_out_user_vectors, mmap_mode='r')

    if matrix.shape[0] == total_users_csv and matrix.shape[1] == config.dim:
        logger.info(f"[PASS] Kích thước Ma trận chuẩn xác: {matrix.shape}")
    else:
        logger.warning(f"[FAIL] Kích thước sai lệch: {matrix.shape} vs {(total_users_csv, config.dim)}")

    # Kiểm tra Dữ liệu rỗng (NaN/Inf)
    if np.any(np.isnan(matrix)) or np.any(np.isinf(matrix)):
        logger.error("[FAIL] Phát hiện NaN hoặc Infinity trong Vector!")
    else:
        logger.info("[PASS] Vector toán học sạch sẽ, không có NaN/Inf.")

# Chạy kiểm thử
acceptance_test(UserProfileConfig())

In [8]:
!pip install implicit

  Using cached implicit-0.7.2.tar.gz (70 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Operation cancelled by user
